# Vision Expert 分析图（用于 slide）

生成 4 张图给 paper / 汇报 slide 用：
- **(a) Grad-CAM** — ResNet3D 在 throw 动作上看哪些区域
- **(b) Temporal dynamics** — 不同动作的 token energy + throw 的 token 相似度矩阵
- **(c) Token activation** — Qwen ViT 在 throw 上不同时刻的 token 激活
- **(d) t-SNE** — ResNet3D 特征空间 27 类聚类

修复了 Hang 原版代码里的几个粘贴 bug，路径改成自动搜索。

**前提：** Drive 上有 `rgb_frames_112.pt`(~7 GB) 和 `resnet3d_vision_expert.pt`(~133 MB)。Qwen tokens 如果有更好，没有也能跑前 3 张图。

## 1. 挂 Drive + 找文件

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 自动找需要的 4 个文件

我们要找：
- `rgb_frames_112.pt` (RGB 帧缓存)
- `resnet3d_vision_expert.pt` (ResNet3D 模型权重)
- `vision_tokens_qwen25.pt` (Qwen tokens，可选)
- 一个能写图的 figures 目录

In [ ]:
import os, subprocess

def find_one(name):
    r = subprocess.run(
        ["find", "/content/drive/MyDrive", "-name", name, "-not", "-path", "*/.*"],
        capture_output=True, text=True, timeout=120,
    )
    lines = [l.strip() for l in r.stdout.splitlines() if l.strip()]
    return lines[0] if lines else None

RGB_PATH    = find_one("rgb_frames_112.pt")
RESNET_PATH = find_one("resnet3d_vision_expert.pt")
QWEN_PATH   = find_one("vision_tokens_qwen25.pt")

print(f"RGB cache       : {RGB_PATH or 'NOT FOUND'}")
print(f"ResNet3D ckpt   : {RESNET_PATH or 'NOT FOUND'}")
print(f"Qwen tokens     : {QWEN_PATH or 'NOT FOUND (will skip fig c)'}")

# 保存目录: 放到 repo figures 里 (如果 repo 在 Drive 上)
REPO_FIGURES = "/content/drive/MyDrive/Multi-Modal-AI/project/final/figures/vision_analysis"
SAVE_DIR = REPO_FIGURES if os.path.exists(os.path.dirname(REPO_FIGURES)) else "/content/drive/MyDrive/MMAI/figures/vision_analysis"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"\nSAVE_DIR        : {SAVE_DIR}")

assert RGB_PATH and RESNET_PATH, "需要 rgb_frames_112.pt 和 resnet3d_vision_expert.pt 才能跑"

## 3. 导入 + 定义 ResNet3DExpert

修复了 Hang 原版 forward 里的截断 bug：缺了 `cls_token.expand` + `torch.cat`。

In [ ]:
import torch
import torch.nn as nn
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from sklearn.manifold import TSNE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


class ResNet3DExpert(nn.Module):
    def __init__(self, d_model=256, dropout=0.3):
        super().__init__()
        backbone = torchvision.models.video.r3d_18(
            weights=torchvision.models.video.R3D_18_Weights.DEFAULT)
        self.features = nn.Sequential(
            backbone.stem, backbone.layer1, backbone.layer2,
            backbone.layer3, backbone.layer4)
        self.spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.proj = nn.Sequential(nn.Linear(512, d_model), nn.GELU(), nn.Dropout(dropout))
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 65, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=d_model * 4,
            dropout=dropout, activation='gelu', batch_first=True)
        self.temporal_transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, 27))

    def forward(self, x, return_tokens=False):
        B = x.shape[0]
        feat = self.spatial_pool(self.features(x)).squeeze(-1).squeeze(-1).permute(0, 2, 1)
        T_out = feat.shape[1]
        feat = self.proj(feat)
        cls = self.cls_token.expand(B, -1, -1)
        feat = torch.cat([cls, feat], dim=1)
        feat = feat + self.pos_embed[:, :T_out + 1, :]
        feat = self.temporal_transformer(feat)
        if return_tokens:
            return feat[:, 1:, :]
        return self.head(feat[:, 0, :])

## 4. 加载 RGB 数据 + ResNet3D 权重

`rgb_frames_112.pt` 是 7.24 GB，加载需要 1-2 分钟和充足 RAM（Pro High-RAM 必须）。

In [ ]:
import time
print("Loading rgb_frames_112.pt (~7 GB)...")
t0 = time.time()
rgb_data = torch.load(RGB_PATH, map_location='cpu', weights_only=False)
print(f"  loaded {len(rgb_data)} samples in {time.time()-t0:.1f}s")

print("\nLoading ResNet3D weights...")
model = ResNet3DExpert().to(device)
ckpt = torch.load(RESNET_PATH, map_location=device, weights_only=False)
if isinstance(ckpt, dict) and 'model_state' in ckpt:
    model.load_state_dict(ckpt['model_state'])
    print(f"  ResNet3D loaded (Hang's best_acc = {ckpt.get('best_acc', 'N/A')})")
else:
    model.load_state_dict(ckpt)
    print("  ResNet3D loaded (no metadata)")
model.eval()


def to_uint8(frame):
    f = frame.numpy() if torch.is_tensor(frame) else frame
    if f.max() <= 1.0:
        f = f * 255.0
    return np.clip(f, 0, 255).astype(np.uint8)

## 5. 图 (a): Grad-CAM

挑一个 throw 样本（a5_s1_t1），对 ResNet3D layer3 的输出做 Grad-CAM。把 4 个时间点的热力图叠到 RGB 帧上。

In [ ]:
print("(a) Grad-CAM...")

activation, gradient = {}, {}
def fwd_hook(m, inp, out):
    activation['v'] = out.detach()
def bwd_hook(m, gi, go):
    gradient['v'] = go[0].detach()

h1 = model.features[3].register_forward_hook(fwd_hook)
h2 = model.features[3].register_full_backward_hook(bwd_hook)

try:
    key = (5, 1, 1)  # throw, subject 1, trial 1
    frames = rgb_data[key].float().permute(3, 0, 1, 2).unsqueeze(0).to(device)
    frames.requires_grad_(True)
    output = model(frames)
    pred = output.argmax(1).item()
    model.zero_grad()
    output[0, pred].backward()

    grads = gradient['v']                       # (1, C, T', H', W')
    acts  = activation['v']
    weights = grads.mean(dim=[2, 3, 4], keepdim=True)
    cam = torch.relu((weights * acts).sum(dim=1, keepdim=True))
    cam = cam.squeeze().detach().cpu().numpy()  # (T', H', W')
finally:
    h1.remove(); h2.remove()

T_cam = cam.shape[0]
cam_to_frame = np.linspace(0, 59, T_cam, dtype=int)
cam_indices  = np.linspace(0, T_cam - 1, 4, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ci in enumerate(cam_indices):
    fi = cam_to_frame[ci]
    orig = to_uint8(rgb_data[key][fi])
    cam_s = cam[ci]
    cam_n = (cam_s - cam_s.min()) / (cam_s.max() - cam_s.min() + 1e-8)
    cam_r = np.array(PILImage.fromarray(cam_n.astype(np.float32)).resize((112, 112), PILImage.BICUBIC))

    axes[0, i].imshow(orig)
    axes[0, i].axis('off')
    axes[0, i].set_title(f'F{fi} (t={fi/59:.2f})', fontsize=11)
    axes[1, i].imshow(orig)
    axes[1, i].imshow(cam_r, cmap='jet', alpha=0.5, vmin=0, vmax=1)
    axes[1, i].axis('off')

axes[0, 0].text(-0.08, 0.5, 'RGB', transform=axes[0, 0].transAxes,
                rotation=90, va='center', fontsize=12, fontweight='bold')
axes[1, 0].text(-0.08, 0.5, 'CAM', transform=axes[1, 0].transAxes,
                rotation=90, va='center', fontsize=12, fontweight='bold')
fig.suptitle(f'Grad-CAM: ResNet3D attention during throw action (pred class {pred + 1})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig_gradcam.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"  ✅ saved: {SAVE_DIR}/fig_gradcam.png")

## 6. 图 (b): Temporal dynamics

左图：5 个动作的 token energy 沿时间变化（throw / shoot / bowling / baseball / swipe left）
右图：throw 的 token-token cosine 相似度矩阵

In [ ]:
print("(b) Temporal dynamics...")

# 选 5 个有代表性的动作 (用 Hang 的 action_short label)
action_list = {5: "Throw", 7: "Shoot", 11: "Bowling", 13: "Baseball", 1: "Swipe L"}
colors_map  = {"Throw": "#2D6A9F", "Shoot": "#F59E0B", "Bowling": "#0D9488",
               "Baseball": "#EF4444", "Swipe L": "#8B5CF6"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：每个动作的 token energy
for act, name in action_list.items():
    k = (act, 1, 1)
    if k not in rgb_data:
        continue
    x = rgb_data[k].float().permute(3, 0, 1, 2).unsqueeze(0).to(device)
    with torch.no_grad():
        tok = model(x, return_tokens=True).squeeze().cpu()
    energy = tok.norm(dim=1).numpy()
    e_norm = (energy - energy.mean()) / (energy.std() + 1e-8)
    t = np.linspace(0, 1, len(e_norm))
    axes[0].plot(t, e_norm, label=name, color=colors_map[name], linewidth=1.8, alpha=0.9)

axes[0].set_title('(a) Temporal token energy by action', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Normalized time')
axes[0].set_ylabel('Normalized energy')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.2)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# 右：throw 的 token cosine similarity 矩阵
k = (5, 1, 1)
x = rgb_data[k].float().permute(3, 0, 1, 2).unsqueeze(0).to(device)
with torch.no_grad():
    tok = model(x, return_tokens=True).squeeze().cpu()
tok_n = tok / (tok.norm(dim=1, keepdim=True) + 1e-8)
sim = (tok_n @ tok_n.T).numpy()

im = axes[1].imshow(sim, cmap='magma',
                    vmin=np.percentile(sim, 3), vmax=np.percentile(sim, 99.5))
axes[1].set_title('(b) Throw: inter-token cosine similarity', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Token index')
axes[1].set_ylabel('Token index')
axes[1].set_aspect('equal')
plt.colorbar(im, ax=axes[1], shrink=0.85, label='Cosine similarity')

fig.suptitle('Temporal Feature Dynamics (ResNet3D Vision Expert)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig_temporal.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"  ✅ saved: {SAVE_DIR}/fig_temporal.png")

## 7. 图 (c): Qwen ViT token activation（可选，要 12.6 GB Qwen cache）

如果你 Drive 有 Qwen tokens 缓存，会画 4 个时刻的 token-特征热力图（红蓝双色，红=正激活）。如果没有，会跳过这张图。

In [ ]:
print("(c) Token activation...")

if QWEN_PATH:
    from matplotlib.colors import TwoSlopeNorm

    print(f"  loading Qwen cache (~13 GB)... 慢")
    vision_cache = torch.load(QWEN_PATH, map_location='cpu', weights_only=False)
    tokens = vision_cache[(5, 1, 1)]   # throw, shape (60, 64, 2048)

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
    times = [0, 20, 40, 59]
    for i, t in enumerate(times):
        data = tokens[t, :, :160].numpy()   # 64 tokens × 前 160 维
        vmax = np.percentile(np.abs(data), 98)
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        axes[i].imshow(data, aspect='auto', cmap='RdBu_r', norm=norm, interpolation='nearest')
        axes[i].set_title(f't = {t}', fontsize=12, fontweight='bold')
        axes[i].set_xlabel('Feature dim.')
        if i == 0:
            axes[i].set_ylabel('Vision token')
            axes[i].set_yticks([0, 32, 63])
        else:
            axes[i].set_yticks([])

    fig.suptitle('Vision-Token Activation Snapshots (Qwen2.5-VL, throw action)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/fig_token_activation.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"  ✅ saved: {SAVE_DIR}/fig_token_activation.png")

    # 释放 12 GB 内存
    del vision_cache
    import gc; gc.collect()
else:
    print("  ⚠ Qwen cache 不存在, 跳过 fig (c)")

## 8. 图 (d): t-SNE

跑 860 个样本的 ResNet3D 特征 → mean over T → t-SNE 2D。1-2 分钟。

In [ ]:
print("(d) t-SNE... (~1-2 分钟)")

features_list, labels_list = [], []
t0 = time.time()
with torch.no_grad():
    for i, ((action, subject, trial), frames) in enumerate(rgb_data.items()):
        x = frames.float().permute(3, 0, 1, 2).unsqueeze(0).to(device)
        tok = model(x, return_tokens=True).squeeze().cpu()
        features_list.append(tok.mean(dim=0).numpy())
        labels_list.append(action)
        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/{len(rgb_data)}  ({time.time()-t0:.0f}s)")

features_np = np.array(features_list)
labels_np   = np.array(labels_list)

try:
    tsne = TSNE(n_components=2, perplexity=30, random_state=42,
                init='pca', learning_rate='auto', max_iter=1000)
except TypeError:
    tsne = TSNE(n_components=2, perplexity=30, random_state=42,
                init='pca', learning_rate='auto', n_iter=1000)

embedded = tsne.fit_transform(features_np)

action_short = {
    1:  'Swipe L', 2:  'Swipe R', 3:  'Wave',     4:  'Clap',     5:  'Throw',
    6:  'Arm cross', 7: 'Shoot',  8:  'Draw X',   9:  'Draw O',  10:  'Draw tri',
    11: 'Bowling', 12: 'Boxing', 13: 'Baseball', 14: 'Tennis',  15: 'Arm curl',
    16: 'Serve',   17: 'Push',   18: 'Knock',    19: 'Catch',   20: 'Pickup',
    21: 'Sit→Stand', 22: 'Stand→Sit', 23: 'Lunge', 24: 'Squat', 25: 'Kick',
    26: 'Walk',    27: 'Jog',
}

fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.get_cmap('tab20', 27)

for act in range(1, 28):
    mask = labels_np == act
    if mask.sum() == 0:
        continue
    ax.scatter(embedded[mask, 0], embedded[mask, 1], s=20, alpha=0.75,
               color=cmap(act - 1), edgecolors='none',
               label=f'{act}: {action_short.get(act, act)}')
    cx = np.median(embedded[mask, 0])
    cy = np.median(embedded[mask, 1])
    ax.text(cx, cy, str(act), fontsize=7, fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.7))

ax.set_title('t-SNE: ResNet3D Feature Space (89.3% accuracy)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE dim. 1')
ax.set_ylabel('t-SNE dim. 2')
ax.legend(fontsize=7, ncol=3, loc='center left', bbox_to_anchor=(1.02, 0.5),
          frameon=True, framealpha=0.9, edgecolor='#ddd')
ax.grid(True, alpha=0.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig_tsne.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"  ✅ saved: {SAVE_DIR}/fig_tsne.png")

## 9. 拼成 2×2 合图

In [ ]:
import matplotlib.image as mpimg

panels = [
    (f'{SAVE_DIR}/fig_gradcam.png',          '(a) Grad-CAM'),
    (f'{SAVE_DIR}/fig_temporal.png',         '(b) Temporal dynamics'),
    (f'{SAVE_DIR}/fig_token_activation.png', '(c) Token activation (Qwen ViT)'),
    (f'{SAVE_DIR}/fig_tsne.png',             '(d) t-SNE clustering'),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, (path, title) in zip(axes.flat, panels):
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
    else:
        ax.text(0.5, 0.5, f'{title}\n(skipped)', transform=ax.transAxes,
                ha='center', va='center', fontsize=14, color='gray')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

fig.suptitle('Vision Expert Analysis', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig_analysis_combined.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"\n✅ all figures saved under {SAVE_DIR}")

## 完事

`SAVE_DIR` 下应该有 5 个 png：
- `fig_gradcam.png`
- `fig_temporal.png`
- `fig_token_activation.png` (如果 Qwen cache 存在)
- `fig_tsne.png`
- `fig_analysis_combined.png` (2×2 合图)

直接拿来插 slide。